# GEF2017: all methods and hierarchy consistency checks

This entry point retains its original filename and analyzes the current project's 10-node deduplicated hierarchy.
It covers FHF, HFF, HFFPS, 1/3/6-step horizons, and all available reconciliation methods.
Audit per-node accuracy, structural levels, semantic evaluation levels, parent-child consistency, and input provenance separately.

In [ ]:
# Locate the project package from a notebook folder or the project root.
from pathlib import Path
import os
import sys
_pphf_start = Path.cwd().resolve()
_pphf_override = os.environ.get("PPHF_PROJECT_ROOT") or os.environ.get("DP_FHF_PROJECT_ROOT")
_pphf_candidates = ([Path(_pphf_override).expanduser()] if _pphf_override else []) + [_pphf_start, *_pphf_start.parents]
PROJECT_ROOT = next((p.resolve() for p in _pphf_candidates if (p / "PPHF_modules" / "fhf" / "FHF_runner.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate PPHF_modules. Set PPHF_PROJECT_ROOT or start Jupyter inside the project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ROOT = PROJECT_ROOT
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from PPHF_modules.results_check.current_results import (
    prepare_current_results, core_accuracy_table, plot_training, plot_timing,
)

In [ ]:
DATASET = 'gef2017'
SOURCE_DIR = ROOT / 'fhf_dedup_gef2017'
OUTPUT_DIR = ROOT / 'Results_check' / 'deduplication_results' / DATASET
RECOMPUTE = False
tables = prepare_current_results(DATASET, root=ROOT, source_dir=SOURCE_DIR,
                                 output_dir=OUTPUT_DIR, force=RECOMPUTE)

## Nodes and levels

Use structural levels for parent-child consistency and semantic evaluation levels for accuracy summaries.
A leaf retained after deduplication may be at structural level 2 while belonging to the bottom level for accuracy evaluation.

In [ ]:
nodes = tables['per_node'][['unique_id','structural_level','level','role']].drop_duplicates()
display(nodes.sort_values(['structural_level','unique_id']))
display(tables['hierarchy_audit'])

## Accuracy across all methods and per-node results

In [ ]:
display(tables['accuracy_by_level'])
display(tables['per_node'].head(40))
print('All node metrics:', OUTPUT_DIR / 'per_node.csv')

## Parent-child constraints and normalized consistency

In [ ]:
display(tables['coherence_by_method'])
parents = tables['coherence_by_parent']
display(parents[['horizon','source_system','method_internal','parent_id','level',
                 'incoherence_total','mase_scale','MASI','n_time_points']])

## Numerical tolerance checks

Compare the MASI of each reconciliation method with the specified tolerance and retain each diagnostic result.
Assess forecast accuracy using the MAE/MASE/RMSSE tables above.

In [ ]:
COHERENCE_TOLERANCE = 1e-6
checks = tables['coherence_by_method'].loc[
    ~tables['coherence_by_method'].method_internal.isin(['base','naive'])].copy()
checks['within_tolerance'] = checks.MASI <= COHERENCE_TOLERANCE
display(checks[['horizon','source_system','method_internal','MASI','within_tolerance']])
checks.to_csv(OUTPUT_DIR / 'coherence_checks.csv', index=False)

## Input provenance and content fingerprints

In [ ]:
manifest = json.loads((OUTPUT_DIR / 'source_manifest.json').read_text())
display(pd.DataFrame(manifest['signature']['sources']))
display(pd.DataFrame([manifest['validation']]))
assert set(tables['hierarchy_audit'].nodes) == {10}
assert set(tables['hierarchy_audit'].horizon) == {1,3,6}
assert set(tables['hierarchy_audit'].source_system) == {'FHF','HFF','HFFPS'}